In [7]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from pydantic import BaseModel
import gradio as gr

load_dotenv()

True

In [8]:
# system_message = """You are a helpful assistant that translates natural language to SQL queries to execute. You will be given a natural language query 
# and a list of tables details in the database with business details. You'll be expected find the intent of the user request based on the tablecontext given below 
# and generate a SQL query that accurately address the needs with human understanding explanation, while adhering to the business logic and constraints provided.
# table_context: {loaded_intent} 
# """

system_message = """
You are an expert Text-to-SQL and data analytics assistant for a quick-commerce platform.
Your job is to read the user's natural language question, look at the provided Semantic Schema YAML manifest, and generate an accurate, safe MySQL query.

### RULES:
1. Only generate valid MySQL queries (SELECT statements only - no DROP, DELETE, or UPDATE).
2. Use the entities, dimensions, and measures defined in the YAML manifest to resolve table joins and column names.
3. If a metric or dimension is ambiguous, ask for clarification instead of guessing.
4. Output your response in a clean JSON format containing the explanation and the SQL query.

### SEMANTIC SCHEMA MANIFEST (YAML):
{loaded_intent}
"""

In [9]:
load_dotenv(override=True)

openai_api_key = os.getenv('GROQ_API_KEY')
# if openai_api_key:
#     print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
# else:
#     print("OpenAI API Key not set")
    
MODEL = "openai/gpt-oss-120b"
groq_url = "https://api.groq.com/openai/v1"
openai = OpenAI(base_url=groq_url,api_key=openai_api_key)

In [10]:
def load_semantic_manifest(file_path: str):
    with open(file_path, "r") as file:
        # We can load it as raw text to keep formatting intact for the prompt
        return file.read()

In [11]:
def call_llm(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", 
                 "content": system_message.format(loaded_intent=load_semantic_manifest("intents/tables_intent_20261021.yaml"))}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2
        )
    return response.choices[0].message.content

In [12]:
gr.ChatInterface(fn=call_llm).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
